# Evaluación de CerebraIA: FID, KID y verificación con clasificador

Ejecutar en **Google Colab con GPU T4** (Entorno de ejecución → Cambiar tipo → T4 GPU).

Requisitos previos:
- Secrets de Colab: `KAGGLE_USERNAME` y `KAGGLE_KEY` (solo para descargar el dataset).
- El generador entrenado (`generator_final.keras`) en Google Drive.

Las imágenes son sintéticas y de uso educativo; esto no reemplaza el diagnóstico médico.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "SIN GPU: activa la T4")

## 1. Código y dependencias

In [ ]:
REPO_URL = "https://github.com/Dgarzonac9/CerebraIA"
BRANCH = "main"   # cambia a la rama de trabajo si aún no se fusionó

!rm -rf /content/CerebraIA
!git clone --depth 1 -b {BRANCH} {REPO_URL} /content/CerebraIA
%cd /content/CerebraIA
!pip install -q scipy scikit-learn pandas kaggle

## 2. Dataset real (Kaggle)

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

DATA_DIR = "/content/dataset"
!kaggle datasets download -d ishans24/brain-tumor-dataset -p {DATA_DIR} --unzip
!find {DATA_DIR} -maxdepth 2 -type d | head -20

## 3. Generador entrenado

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WEIGHTS = "/content/drive/MyDrive/Practica_3_IA/modelos/generator_final.keras"
assert os.path.exists(WEIGHTS), f"No se encontró {WEIGHTS}"

## 4. FID y KID (global)

El generador es **incondicional** (solo recibe ruido), por lo que no hay métrica por clase.
Con `N_SAMPLES >= 2048` la covarianza de las características de InceptionV3 deja de ser singular y el FID es estable.

In [ ]:
N_SAMPLES = 5000
!python -m eval.quality_metrics --weights {WEIGHTS} --data-dir {DATA_DIR} --n-samples {N_SAMPLES} --out eval/results/metrics.json

## 5. Verificación con clasificador

CNN entrenada solo con imágenes reales; luego clasifica imágenes sintéticas para medir reconocibilidad y diversidad (mode collapse).

In [ ]:
!python -m eval.classifier_check --weights {WEIGHTS} --data-dir {DATA_DIR} --n-samples 1000 --epochs 15 --out eval/results/classifier.json

## 6. Tablas de resultados

In [ ]:
import json
import pandas as pd

m = json.load(open("eval/results/metrics.json"))
c = json.load(open("eval/results/classifier.json"))

display(pd.DataFrame([{
    "n_real": m["n_real"], "n_sintéticas": m["n_sinteticas"],
    "FID": round(m["global"]["fid"], 2),
    "KID (media)": round(m["global"]["kid_mean"], 5),
    "KID (desv.)": round(m["global"]["kid_std"], 5),
}]).set_index("n_real"))

display(pd.DataFrame([{
    "accuracy val (reales)": round(c["val_accuracy"], 4),
    "confianza media (sintéticas)": round(c["sinteticas"]["confianza_media"], 4),
    "entropía normalizada": round(c["sinteticas"]["entropia_normalizada"], 4),
    "posible mode collapse": c["sinteticas"]["posible_mode_collapse"],
}]))

display(pd.Series(c["sinteticas"]["distribucion_predicha"], name="proporción predicha").round(4).to_frame())

## 7. Guardar resultados

Copia `eval/results/*.json` a Drive o envíalos para completar la tabla del README.

In [ ]:
!mkdir -p /content/drive/MyDrive/Practica_3_IA/eval && cp eval/results/*.json /content/drive/MyDrive/Practica_3_IA/eval/